# Forecast

Predict future energy consumption.

In [0]:
%pip install xgboost

In [0]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from pyspark.sql import functions as F
import numpy as np
from xgboost import XGBRegressor

START_YEAR = 2020
GOLD_HOURLY_PATH = "/Volumes/workspace/default/gold/demand_hourly"
GOLD_FORECAST_PATH = "/Volumes/workspace/default/gold/demand_forecast"
HOURS_PER_DAY = 24
DAYS_PER_WEEK = 7
MONTHS_PER_YEAR = 12
FORECAST_YEAR = (spark.read
    .format("delta")
    .load(GOLD_HOURLY_PATH)
    .agg(F.max("year"))
    .collect()[0][0])

print(f"[ML] forecast year: {FORECAST_YEAR}")  # may be affected by 2026-01-01

def load():
    return (spark.read
            .format("delta")
            .load("/Volumes/workspace/default/gold/demand_hourly")
            .toPandas())

def add_lag_features(pdf):
    pdf = pdf.sort_values("hour_ts")
    
    # lag features
    pdf["lag_day"] = pdf["avg_demand_mw"].shift(HOURS_PER_DAY)
    pdf["lag_week"] = pdf["avg_demand_mw"].shift(HOURS_PER_DAY * DAYS_PER_WEEK)
    pdf["rolling_week_avg"] = pdf["avg_demand_mw"].rolling(HOURS_PER_DAY * DAYS_PER_WEEK).mean()  # 7 day rolling avg

    pdf = pdf.dropna()  # remove rows where lags don't exist yet
    return pdf

def add_cyclic_features(df):
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / HOURS_PER_DAY)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / HOURS_PER_DAY)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / MONTHS_PER_YEAR)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / MONTHS_PER_YEAR)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / DAYS_PER_WEEK)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / DAYS_PER_WEEK)
    return df

def train_and_predict(df):
    features = [
        "hour_sin", "hour_cos",
        "month_sin", "month_cos",
        "dow_sin", "dow_cos",
        "quarter", "is_weekend",
        "avg_price_rrp",
        "lag_day", "lag_week", "rolling_week_avg"
    ]

    target = "avg_demand_mw"

    train = df[df["year"] < FORECAST_YEAR]  # train on all years before forecast year
    test = df[df["year"] == FORECAST_YEAR]  # test on forecast year

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        random_state=42
    )

    model.fit(train[features], train[target])

    test = test.copy()
    test["predicted_demand_mw"] = model.predict(test[features])
    train = train.copy()
    train["predicted_demand_mw"] = model.predict(train[features])

    mae = mean_absolute_error(test[target], test["predicted_demand_mw"])
    rmse = root_mean_squared_error(test[target], test["predicted_demand_mw"])

    print(f"[ML] forecast year: {FORECAST_YEAR}")
    print(f"[ML] training years: {FORECAST_YEAR - START_YEAR} ({START_YEAR} to {FORECAST_YEAR - 1})")
    print(f"[ML] mae: {mae:.2f} mw")
    print(f"[ML] rmse: {rmse:.2f} mw")

    return pd.concat([train, test])

def write_forecast(df):
    df_forecast = spark.createDataFrame(df[[
        "hour_ts", "region", "avg_demand_mw", "predicted_demand_mw"
    ]].rename(columns={"avg_demand_mw": "actual_demand_mw"}))

    df_forecast = df_forecast.withColumn("created_at", F.current_timestamp())

    (df_forecast.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(GOLD_FORECAST_PATH))
    print("[ML] forecast written to gold")

write_forecast(
    train_and_predict(
        add_cyclic_features(
            add_lag_features(
                load()
            )
        )
    )
)